# Running zagg from a JupyterHub

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/englacial/zagg/main?urlpath=lab/tree/notebooks/jupyterhub_example.ipynb)

This notebook shows how to drive the `zagg` Python API from a Jupyter
notebook (e.g. a science JupyterHub such as CryoCloud or Pangeo, or your own
laptop). It covers **local processing** -- running the aggregation on the hub
itself with threads, no AWS Lambda required.

> **Lambda fan-out** for full-scale runs lives in the
> [cryocloud_example](cryocloud_example.ipynb) notebook, which dispatches to a
> deployed AWS Lambda and therefore cannot run on Binder.

### What runs where

| Section | Needs credentials? | Runs on Binder? |
|---------|--------------------|-----------------|
| Build a catalog (CMR-STAC metadata query) | none -- the CMR-STAC search is anonymous | **yes** |
| Local `agg()` -- fetch granule pixels from NSIDC | **NASA Earthdata login** | no |
| Read & visualize a published result | none (anonymous source.coop) | **yes** |

Building the catalog is a pure **metadata** query against CMR-STAC and needs
**no login** -- it runs on Binder (the HEALPix `mortie` coverage backend is the
default, so the non-PyPI [spherely fork](https://github.com/espg/spherely) is
*not* required either). Only *fetching the granule pixels* -- the `agg()` step
that reads ICESat-2 HDF5 from NSIDC -- needs a NASA Earthdata login, so those
cells are written to **skip cleanly** when credentials are absent. The final
section reads a public, anonymous result and produces the plots end-to-end.

## 1. Authentication

zagg needs two sets of credentials:

- **NASA Earthdata** -- for reading source data (ICESat-2 HDF5 on NSIDC S3)
- **AWS** (optional) -- for writing output to S3 or invoking Lambda

Both use standard credential discovery -- if you already have `~/.netrc`
and `~/.aws/credentials` configured, skip this section entirely.

### Default credential locations

| Service | File | Format |
|---------|------|--------|
| Earthdata | `~/.netrc` | `machine urs.earthdata.nasa.gov login USERNAME password PASSWORD` |
| AWS | `~/.aws/credentials` | `[default]` profile with `aws_access_key_id` / `aws_secret_access_key` |

### Operator-managed hub (CryoCloud, Pangeo)

Credentials are pre-configured as environment variables or mounted files.
Nothing to do -- `earthaccess` and `boto3` discover them automatically.

### Override via environment variables

Only needed if the standard files above are not present:

In [ ]:
import os

# Only needed if ~/.netrc is not configured:
# os.environ["EARTHDATA_USERNAME"] = "your_username"
# os.environ["EARTHDATA_PASSWORD"] = "your_password"

# Only needed if ~/.aws/credentials is not configured:
# os.environ["AWS_ACCESS_KEY_ID"] = "AKIA..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_DEFAULT_REGION"] = "us-west-2"

## 2. Load a pipeline config

The config defines what data to read, how to aggregate it, and where to
write output. Use the built-in ATL06 config or load a custom YAML.

In [ ]:
from zagg import default_config, get_child_order

config = default_config("atl06")

print(f"Reader: {config.data_source['reader']}")
print(f"Groups: {config.data_source['groups']}")
print(f"Variables: {list(config.data_source['variables'].keys())}")
print(f"Child order: {get_child_order(config)}")

## 3. Build a catalog *(anonymous -- runs on Binder)*

The catalog (a *ShardMap*) maps grid cells to granule hrefs. It is built from a
**CMR-STAC metadata query**, which is anonymous: no NASA Earthdata login is
needed to *find* the granules (only *fetching* their pixels in section 4 needs
auth). The build is driven by the **same config** as the run so the shard grid
can't drift from the output grid, and the default HEALPix `mortie` coverage
backend means the non-PyPI [spherely fork](https://github.com/espg/spherely) is
not required here.

The cell below builds a small Antarctic catalog over a short date window so it
finishes quickly on Binder. Equivalently from the CLI:

```bash
# Bundled atl06 config; small bbox + a few days so the build is quick:
python -m zagg.catalog --config atl06.yaml --short-name ATL06 --version 006 \
    --start-date 2025-04-01 --end-date 2025-04-03 \
    --bbox=-70,-78,-65,-75 --backend mortie
# -> writes shardmap_ATL06_<start>_<end>.json
```

(For a full cycle, swap the bbox/dates for `--cycle 22 --polygon antarctica.geojson`.)

In [ ]:
# Build a small ATL06 catalog anonymously (CMR-STAC metadata only -- no login).
# This is the same work `python -m zagg.catalog ...` does, via the Python API.
from zagg.catalog.shardmap import ShardMap
from zagg.catalog.sources import CMRSource, Query
from zagg.grids import from_config

catalog_path = "shardmap_ATL06_2025-04-01_2025-04-03.json"
grid = from_config(config)

# Small bbox + a few days so the build is quick on Binder.
query = Query(
    short_name="ATL06",
    version="006",
    start_date="2025-04-01",
    end_date="2025-04-03",
    region=(-70.0, -78.0, -65.0, -75.0),  # lon_min, lat_min, lon_max, lat_max
)

try:
    cat = CMRSource().fetch(query)
    print(f"Fetched {len(cat)} granules ({query.collection})")
    # backend="mortie": HEALPix coverage, so no spherely fork needed.
    sm = ShardMap.build(cat, grid, backend="mortie")
    sm.to_json(catalog_path)
    HAVE_CATALOG = True
    print(f"ShardMap: {len(sm.shard_keys)} shards, "
          f"{sm.metadata['total_pairs']} pairs -> {catalog_path}")
except Exception as exc:
    # CMR-STAC is anonymous but needs outbound network; degrade cleanly on an
    # offline / egress-restricted Binder so the rest of the notebook still runs.
    HAVE_CATALOG = False
    print(f"Catalog build skipped ({type(exc).__name__}: {exc}).\n"
          "If this environment can't reach cmr.earthdata.nasa.gov, jump to the "
          "public-result section, which reads source.coop directly.")

## 4. Local processing *(requires NASA Earthdata login)*

The catalog above is just metadata; **fetching the granule pixels** is what
needs a NASA Earthdata login. Process cells locally with a thread pool -- no
Lambda needed. Good for testing, small regions, or hubs without AWS set up.
`driver="https"` reads ICESat-2 HDF5 over HTTPS (works from anywhere, still
needs Earthdata auth); `driver="s3"` is faster but only from `us-west-2`.

These cells run only when `HAVE_CATALOG` is true (i.e. the catalog built above)
**and** an Earthdata login is configured; without credentials they skip cleanly.

In [ ]:
from zagg import agg

if HAVE_CATALOG:
    # Dry run -- preview the plan without processing.
    preview = agg(
        config,
        catalog=catalog_path,
        store="./test_output.zarr",
        driver="https",
        dry_run=True,
    )
    print(preview)
else:
    print("Skipped: no catalog (needs an Earthdata login).")

In [ ]:
import logging

if HAVE_CATALOG:
    logging.basicConfig(level=logging.INFO, format="%(message)s")
    # Process 2 cells locally, write to a local Zarr. driver="https" works
    # outside us-west-2.
    results = agg(
        config,
        catalog=catalog_path,
        store="./test_output.zarr",
        driver="https",
        max_cells=2,
        max_workers=2,
        overwrite=True,
    )
    print(f"Cells with data:    {results['cells_with_data']}")
    print(f"Total observations: {results['total_obs']:,}")
    print(f"Wall time:          {results['wall_time_s']:.1f}s")
else:
    print("Skipped: no catalog (needs an Earthdata login).")

## 5. Inspect the local output

The output is a Zarr v3 store following the DGGS convention. (Runs only if the
local processing above produced `./test_output.zarr`.)

In [ ]:
import numpy as np
import zarr

if HAVE_CATALOG:
    store = zarr.open_group("./test_output.zarr", mode="r")
    child_order = get_child_order(config)
    group = store[str(child_order)]

    print(f"Arrays: {list(group.array_keys())}")
    for nm in ["count", "h_mean"]:
        arr = group[nm]
        print(f"  {nm}: shape={arr.shape}, dtype={arr.dtype}")

    count = group["count"][:]
    has_data = count > 0
    print(f"\nCells with data: {has_data.sum():,} / {len(count):,}")
    print(f"Total observations: {count[has_data].sum():,}")
else:
    print("Skipped: no local output to inspect.")

## 6. Read & visualize a published result *(no credentials -- runs on Binder)*

The `agg()` fetch above needs an Earthdata login. This section instead reads a
**public, anonymous** ATL06 cycle-22 aggregation published on
[source.coop](https://source.coop/englacial/zagg/benchmarks)
(`englacial/zagg/benchmarks/atl06_cycle22_fullsphere.zarr`), so it runs end-to-end on Binder.

We open the HEALPix order-12 store read-only over an unsigned S3 request, then
scatter the filled cells on an Antarctic Polar Stereographic map.

In [ ]:
import numpy as np
import xarray as xr
from obstore.store import S3Store
from zarr.storage import ObjectStore

# Public source.coop store -- unsigned (anonymous) read works from anywhere.
PUBLIC_BUCKET = "us-west-2.opendata.source.coop"
PUBLIC_PREFIX = "englacial/zagg/benchmarks/atl06_cycle22_fullsphere.zarr"
CHILD_ORDER = 12

s3 = S3Store(
    PUBLIC_BUCKET,
    prefix=PUBLIC_PREFIX,
    region="us-west-2",
    skip_signature=True,
)
pub = xr.open_dataset(
    ObjectStore(store=s3, read_only=True),
    engine="zarr",
    consolidated=True,
    zarr_format=3,
    group=str(CHILD_ORDER),
)

has_data = pub["count"].values > 0
cell_ids = pub["cell_ids"].values[has_data].astype(np.int64)
h_mean = pub["h_mean"].values[has_data]
count = pub["count"].values[has_data].astype(np.float64)
print(f"Filled HEALPix cells: {has_data.sum():,} / {has_data.size:,}")
print(f"Total observations:   {count.sum():.0f}")

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
from mortie._healpix import pix2ang

# HEALPix cell centers -> lon/lat (NESTED), let cartopy project to polar stereo.
# pix2ang takes the HEALPix order (depth), not nside.
lon, lat = pix2ang(CHILD_ORDER, cell_ids)

proj = ccrs.SouthPolarStereo()
fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={"projection": proj})
for ax in axes:
    ax.coastlines(resolution="50m", linewidth=0.5)
    ax.add_feature(cfeature.LAND, facecolor="lightgray", alpha=0.3)
    ax.gridlines(draw_labels=False, alpha=0.3)
    ax.set_extent([-180, 180, -90, -60], crs=ccrs.PlateCarree())

im = axes[0].scatter(lon, lat, c=h_mean, s=0.05, cmap="terrain", vmin=0, vmax=4000,
                     transform=ccrs.PlateCarree())
axes[0].set_title(f"Mean Elevation ({has_data.sum():,} cells)")
plt.colorbar(im, ax=axes[0], label="m", shrink=0.7)

im = axes[1].scatter(lon, lat, c=count, s=0.05, cmap="viridis",
                     norm=plt.matplotlib.colors.LogNorm(vmin=1),
                     transform=ccrs.PlateCarree())
axes[1].set_title("Observation Count")
plt.colorbar(im, ax=axes[1], label="count (log)", shrink=0.7)

plt.suptitle("ATL06 cycle 22 (published source.coop result)", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

## 7. Custom configs

You can embed defaults (catalog path, store path) directly in the config object
so you don't repeat them in every `agg()` call, or load a fully custom YAML with
different variables and aggregation functions.

See the [custom_aggregations](custom_aggregations.ipynb) notebook for examples
of modifying statistics and adding new input variables.

In [ ]:
from zagg import load_config

# Load from a custom YAML file:
# config = load_config("my_custom_config.yaml")

# Or set catalog/store in the config itself:
config.catalog = catalog_path
config.output["store"] = "./output.zarr"

# Then agg() uses them as defaults (CLI/kwargs still override):
# results = agg(config, max_cells=5)